In [1]:
#Does the average Body Mass Index (BMI) of players differ significantly between teams that advanced to the knockout stage (advanced) and teams that were eliminated in the group stage (eliminated) in the FIFA World Cup 2026?

#Data Sources & Documentation
#FB Ref (Primary Source for Player Physical Metrics):** https://fbref.com/en/
#FIFA Official Website (Tournament Statistics):** https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/statistics
#The Stats Don't Lie (Match & Team Outcomes):** https://www.thestatsdontlie.com/football/world-cup-2026/

import pandas as pd
from scipy import stats

In [2]:
#Loaded the final dataset
df = pd.read_csv("FIFAWC2026_Final_Sample.csv")
print(df.head())

              player_name Pos  MP  Min  goals  height_cm  weight_kg      Team  \
0  Alexander Bernhardsson  MF   3  217      0        185         69    Sweden   
1           Jules KoundÃ©  DF   8  599      0        178         83    France   
2              Malo Gusto  DF   5  121      0        179         69    France   
3            Julio Enciso  FW   5  386      1        168         63  Paraguay   
4             Matt Turner  GK   1   90      0        191         83       USA   

  team_status        BMI  
0    advanced  20.160701  
1    advanced  26.196187  
2    advanced  21.534908  
3    advanced  22.321429  
4    advanced  22.751569  


In [3]:
#Filter valid groups
#Count player in advanced group
#Count player in eliminated group
n_advanced_pop = len(df[df['team_status'] == 'advanced'])
n_eliminated_pop = len(df[df['team_status'] == 'eliminated'])

#Set target sample
target_sample_size = 50

#Adjust sample size if population is smaller than the target
sample_size = min(target_sample_size, n_advanced_pop, n_eliminated_pop)

if sample_size < target_sample_size:
    print(f"[Warning] Limited data available. Adjusting sample size per group to: {sample_size}")

#Split the populations
df_advanced_pop = df[df['team_status'] == 'advanced']
df_eliminated_pop = df[df['team_status'] == 'eliminated']

#Simple Random Sampling
sample_advanced = df_advanced_pop.sample(n=sample_size, random_state=42)
sample_eliminated = df_eliminated_pop.sample(n=sample_size, random_state=42)

#Test for equality of variances (Levene's Test)
levene_stat, levene_p = stats.levene(sample_advanced['BMI'], sample_eliminated['BMI'])
print("Levene's Test for Equality of Variances:")
print(f"  - Test Statistic: {levene_stat:.4f}")
print(f"  - p-value: {levene_p:.4f}")

Levene's Test for Equality of Variances:
  - Test Statistic: 0.2235
  - p-value: 0.6374


In [4]:
# Select test configuration based on Levene's Test result
if levene_p < 0.05:
    print("  -> Conclusion: Population variances are significantly different. Running Welch's t-test.")
    equal_var_setting = False
else:
    print("  -> Conclusion: No significant difference in variances. Running standard Student's t-test.")
    equal_var_setting = True

#Execute Two-Sample t-Test
t_stat, t_p = stats.ttest_ind(sample_advanced['BMI'], sample_eliminated['BMI'], equal_var=equal_var_setting)
print("\nTwo-Sample t-Test Results:")
print(f"  - t-statistic: {t_stat:.4f}")
print(f"  - p-value: {t_p:.4f}")


  -> Conclusion: No significant difference in variances. Running standard Student's t-test.

Two-Sample t-Test Results:
  - t-statistic: 1.3691
  - p-value: 0.1741


In [5]:
#Hypotheses conclusion
alpha = 0.05
if t_p < alpha:
    print(f"  -> Conclusion: Reject Null Hypothesis (H0) because p-value ({t_p:.4f}) < {alpha}.")
    print("     There is a statistically significant difference in the average BMI of players "
              "between the 32 advanced teams and the 16 eliminated teams.")
else:
    print(f"  -> Conclusion: Fail to Reject Null Hypothesis (H0) because p-value ({t_p:.4f}) >= {alpha}.")
    print("     There is insufficient statistical evidence to conclude that the average BMI of players "
          "differs significantly between the 32 advanced teams and the 16 eliminated teams.")
print("--------------------------------------------------")

  -> Conclusion: Fail to Reject Null Hypothesis (H0) because p-value (0.1741) >= 0.05.
     There is insufficient statistical evidence to conclude that the average BMI of players differs significantly between the 32 advanced teams and the 16 eliminated teams.
--------------------------------------------------
